## Create 20 atomic models of a 2x2x2 MAPbI3 containing two water molecules randomly displaced and select 10 according to Farthest Point Sampling

In [1]:
from skmatter.sample_selection import FPS
import numpy as np
import pandas as pd
from numpy.linalg import norm
from scipy.spatial.distance import pdist, squareform

#from ase.io import read,write
from ase.visualize import view,ngl
from ase.spacegroup import crystal
#from ase.spacegroup import Spacegroup
from ase.data import atomic_numbers, atomic_names
from ase import Atoms
from ase import neighborlist
from itertools import product

import nglview as nv
from ase.build import molecule

from rascal.representations import SphericalInvariants
#from rascal.models import Kernel

### add missing functions to compute SOAP descriptors checking the notebook TEST_SOAP (points:2)

In [2]:
def get_systems_tag(frames):
    labels = []
    for i, frame in enumerate(frames):
        labels.extend([i]*len(frame))
    return np.array(labels)
def get_dist_mat(soaps_vectors, normalized=True):
    distance = squareform(pdist(soaps_vectors))
    
    if normalized:
        max_val=max(distance.flatten())

    distance_df = pd.DataFrame(distance/max_val)
    # Set display options to show all columns without truncation and maximum 3 decimals
    pd.set_option('display.max_columns', None)
    pd.set_option('display.float_format', lambda x: '%.4f' % x)
    return distance_df
def avg_soaps(atoms_soaps_features, frames):
    df = pd.DataFrame(atoms_soaps_features)
    df["molecule"]=get_systems_tag(frames)
    return df.groupby("molecule").mean().values

def get_kernel_mat(soaps_vectors):
    distance = squareform(pdist(soaps_vectors))
    # Create kernel matrix using Gaussian kernel
    sigma = 0.5  # You can adjust the sigma value according to your requirement
    kernel_matrix = np.exp(-distance ** 2 / (2 * sigma ** 2))

    # Convert kernel matrix to a pandas DataFrame
    kernel_matrix_df = pd.DataFrame(kernel_matrix)
    return kernel_matrix_df

In [3]:
def view_structure(structure,myvec=[]):
    t = nv.ASEStructure(structure)
    w = nv.NGLWidget(t, gui=True)
    w.add_unitcell()
    w.add_ball_and_stick()
    w.add_representation('label',label_type='atomindex',color='black')
    w.add_representation('spacefill',selection=myvec,color="blue",radius=0.5)
    return w

In [4]:
MAPbI3=crystal(
    symbols=['Pb','I','I','N','C','H','H','H','H'],
    basis=[(0.5,0,0),(0.4842,0.25,-0.0562),
           (0.1886,0.0147,0.1844),(0.9421,0.75,0.0297),
           (0.9372,0.25,0.0575),
           (0.9372,0.25,0.1874),(0.8661,0.1701,0.0290),
           (0.1275,0.1891,-0.0085),(0.9543,0.75,0.1459)
          ],
    spacegroup=62,
    cellpar=[8.86574, 12.6293, 8.57689, 90, 90, 90])

In [5]:
#create the 2x2x2 supercell from the unit cell
supercell_no_h2o=MAPbI3.repeat((2,2,2))

In [6]:
#ASE instructions to initialize neighbors list
cutOff = neighborlist.natural_cutoffs(supercell_no_h2o)
nl = neighborlist.NeighborList(cutOff, self_interaction=False, bothways=True)
nl.update(supercell_no_h2o)

#identify NH3 molecules in the crystal
all_N = [atom.index for atom in supercell_no_h2o if atom.symbol == 'N']
all_H_of_N = [index for N in all_N for index in nl.get_neighbors(N)[0] if supercell_no_h2o[index].symbol == 'H'  ]
all_nh3 = all_N + all_H_of_N

In [7]:
### Loop to generate nsamples=20 structures with random positioning of NH2O=2 water molecules

In [8]:
NH2O=2
#dmin and dmax are used to impose some constraints (e.g. distance from N atoms) 
#on the possible positioning of H2O molecules 
nsamples=20
dmin = 1.5
dmax = 2.5
orig_h2o=molecule('H2O')
#transalte the molecule to have Oxygen in (0,0,0)
orig_h2o.translate(-1*orig_h2o.positions[0])
samples=[]
ns=0
while ns < nsamples:
    nh2o = 0
    print("Creating sample ",ns)
    supercell = supercell_no_h2o.copy()
    while nh2o < NH2O:
        h2o = orig_h2o.copy()
        oldcell=supercell.copy()
        t_vector = np.random.uniform(low=-1,high=1,size=(3))
        t_vector = t_vector/np.linalg.norm(t_vector)

        #position h2o within 1.5A---3.0A from a N atom
        t_vector *= np.random.uniform(low=dmin,high=dmax)
        a_random_N = all_N[np.random.randint(low=0,high=len(all_N))]
        t_vector += supercell.positions[a_random_N]
        
        #random rotation of h2o
        r_vector = np.random.uniform(low=-1,high=1,size=(3))
        r_vector = r_vector/np.linalg.norm(r_vector)
        rot_axis=np.random.uniform(low=-1,high=1,size=(3))
        rot_axis = rot_axis/np.linalg.norm(rot_axis) 
        h2o.rotate(np.random.uniform(low=0,high=180),t_vector,center=(0,0,0))
        
        #position h2o
        trial_h2o = h2o.copy()
        trial_h2o.translate(t_vector)
        supercell=supercell + trial_h2o
        natoms=len(supercell) 
        #O of the added h2o molecule is the third last atom: supercell[-3]
        #shortest_O_N_distances=min(supercell.get_distances(supercell[-3].index, all_N, mic=True, vector=False))
        discard=False
        for ih2o,j in product(range(natoms - 3,natoms), range(natoms-3)) :
            if supercell.get_distance(ih2o,j,mic=True,vector=False) < dmin:
                discard = True
                break
        if discard:
            supercell = oldcell.copy()
        else:
            nh2o+=1
    ns+=1
    samples.append(supercell)

Creating sample  0
Creating sample  1
Creating sample  2
Creating sample  3
Creating sample  4
Creating sample  5
Creating sample  6
Creating sample  7
Creating sample  8
Creating sample  9
Creating sample  10
Creating sample  11
Creating sample  12
Creating sample  13
Creating sample  14
Creating sample  15
Creating sample  16
Creating sample  17
Creating sample  18
Creating sample  19


### Explain in few words what we are doing above. This does not look like a Monte Carlo procedure to add H2O molecules, what would we need for implementing MC steps? (points : 2)

In [9]:
for structure in samples:
    structure.wrap()
    structure.pbc=(1,1,1)

In [10]:
hypers = {
    "soap_type":"PowerSpectrum",
    "interaction_cutoff": 5.0,
    "max_radial": 6,
    "max_angular": 6,
    "gaussian_sigma_constant": 0.4,
    "gaussian_sigma_type":"Constant",
    "cutoff_smooth_width":0.5,
    "radial_basis": "GTO",
    "cutoff_function_type": "ShiftedCosine",
    "cutoff_function_parameters":{"width": 0.5},
    "global_species":[1,6,7,53,82]
    }
soap = SphericalInvariants(**hypers)

In [11]:
soap_rep = soap.transform(samples)

### Function to perform FPS using distance matrix)

In [12]:
def FPS(Dmatrix, n=0, idx=None):
    """
        Does Farthest Point Selection on a set of points X
        Adapted from a routine by Michele Ceriotti
    """
    N = Dmatrix.shape[0]

    # If desired number of points less than or equal to zero,
    # select all points
    if n <= 0:
        n = N

    # Initialize arrays to store distances and indices
    fps_idxs = np.zeros(n, dtype=np.int32)
    d = np.zeros(n)

    if idx is None:
        # Pick first point at random
        idx = np.random.randint(0, N)
    fps_idxs[0] = idx

    # Compute distance from all points to the first point
    d1 = Dmatrix[idx]   #np.linalg.norm(X - X[idx], axis=1)**2
    # Loop over the remaining points...
    for i in range(1, n):

        # Get maximum distance and corresponding point
        fps_idxs[i] = np.argmax(d1)
        d[i - 1] = np.amax(d1)

        # Compute distance from all points to the selected point
        d2 =  Dmatrix[fps_idxs[i]]

        # Set distances to minimum among the last two selected points
        d1 = np.minimum(d1, d2)

        if d1.max() == 0.0:
            print("Only {} FPS Possible".format(i))
            return fps_idxs[:i], d[:i]

    return fps_idxs

### explain in few words how we use the distance matrix to do FPS (points: 2)

In [13]:
#compute vector of SOAP features
#group SOAP vectors by molecule and compute average kernel
X=soap_rep.get_features(soap)
avg_soap_samples=avg_soaps(X, samples)
#compute distance matrix (to )
DistMat=get_dist_mat(avg_soap_samples)

In [14]:
#DistMat.values returns the numpy array for the Distance Matrix that was embedded in a pandas object
FPS(DistMat.values,n=10,idx=None)

array([ 7,  4,  8, 13, 12,  9,  6,  2, 16,  3], dtype=int32)

In [15]:
#display DistMat
DistMat

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.0000,0.5459,0.6790,0.5784,0.4689,0.5878,0.7486,0.6337,0.6204,0.5239,0.7338,0.4598,0.8838,0.4691,0.6444,0.6846,0.8097,0.4975,0.7439,0.4616
1,0.5459,0.0000,0.4114,0.5643,0.7601,0.5586,0.7355,0.3795,0.7608,0.5772,0.5566,0.4644,0.6191,0.5664,0.3686,0.5895,0.6015,0.4855,0.5013,0.4861
2,0.6790,0.4114,0.0000,0.6803,0.8117,0.7590,0.9124,0.5208,0.9044,0.5431,0.7458,0.5719,0.7048,0.6969,0.5798,0.7504,0.8234,0.6228,0.6103,0.5474
3,0.5784,0.5643,0.6803,0.0000,0.6681,0.5940,0.4750,0.5075,0.5455,0.4972,0.6059,0.4927,0.6546,0.5093,0.5511,0.4358,0.6675,0.3677,0.5715,0.5228
4,0.4689,0.7601,0.8117,0.6681,0.0000,0.7602,0.8943,0.7898,0.6297,0.5625,0.9004,0.6188,1.0000,0.6535,0.8130,0.8608,0.9948,0.6168,0.9495,0.5804
5,0.5878,0.5586,0.7590,0.5940,0.7602,0.0000,0.5152,0.4681,0.5729,0.7154,0.4553,0.5201,0.7557,0.5420,0.4100,0.4530,0.4310,0.5482,0.6319,0.4654
6,0.7486,0.7355,0.9124,0.4750,0.8943,0.5152,0.0000,0.5821,0.5279,0.7395,0.4719,0.6227,0.7101,0.5788,0.5684,0.2854,0.4761,0.4902,0.5810,0.6457
7,0.6337,0.3795,0.5208,0.5075,0.7898,0.4681,0.5821,0.0000,0.6508,0.6385,0.4198,0.5403,0.5729,0.5796,0.3018,0.4007,0.5171,0.4641,0.4960,0.4567
8,0.6204,0.7608,0.9044,0.5455,0.6297,0.5729,0.5279,0.6508,0.0000,0.6398,0.6515,0.6737,0.9447,0.6922,0.7145,0.5232,0.7363,0.5412,0.8494,0.5524
9,0.5239,0.5772,0.5431,0.4972,0.5625,0.7154,0.7395,0.6385,0.6398,0.0000,0.7017,0.4389,0.7507,0.5693,0.6580,0.6833,0.8157,0.4156,0.6611,0.4776


### Check the list of 10 geometries you obtained from FPS and the matrix DistMat. Does the result make sense? (points: 4)